# Authenticate — Trustless Initiation Proof

> These examples run against Sepolia testnet (`network="sepolia"`).
> To run against Base mainnet: replace `configure(network="sepolia")` with `configure(network="base")`.
> All function calls, record structures, and output shapes are identical across networks.

Every content anchor in AnchorRegistry carries a `tokenCommitment` — a `keccak256(K || arId)` hash stored immutably on Ethereum.

The ownership token `K = keccak256(salt)` is generated in the browser at registration time from 32 uniform random bytes and **never transmitted** to AnchorRegistry servers. Only the token holder can produce the pre-image. This makes the proof trustless: no AnchorRegistry infrastructure needed, no trust required.

Two verification levels:
- **`authenticate_anchor`** — verify a single anchor was initiated by the token holder
- **`authenticate_tree`** — verify tree ownership (Layer 1) + every anchor in the tree (Layer 2)

In [1]:
# Cell 1 — setup
from anchorregistry import configure, authenticate_tree, authenticate_anchor
configure(network="sepolia", rpc_url="https://ethereum-sepolia-rpc.publicnode.com")

In [ ]:
# Cell 2 — authenticate a single anchor
# Replace ownership_token and ar_id with your own values
result = authenticate_anchor(
    ownership_token="0x81fcaa4a0ec65271a4afc92e9ce3a2e4dd7a3c00c9d0d67b41bf417f74e1cfb8",
    ar_id="AR-2026-nPOZvV1"
)
result

In [ ]:
# Cell 3 — authenticate full tree
# Layer 1: keccak256(K || rootArId) == treeId
# Layer 2: verify every user-initiated anchor in the tree
result = authenticate_tree(
    ownership_token="0x81fcaa4a0ec65271a4afc92e9ce3a2e4dd7a3c00c9d0d67b41bf417f74e1cfb8",
    root_ar_id="AR-2026-nPOZvV1"
)
result

In [ ]:
# Cell 4 — cron job audit pattern
# Run this on a schedule to continuously verify your tree hasn't been tampered with
import os
result = authenticate_tree(
    ownership_token=os.environ.get("ANCHOR_OWNERSHIP_TOKEN", "0x81fcaa4a0ec65271a4afc92e9ce3a2e4dd7a3c00c9d0d67b41bf417f74e1cfb8"),
    root_ar_id="AR-2026-nPOZvV1"
)
if not result["authenticated"]:
    print(f"AUDIT FAILED — {result['anchors_failed']} anchors unverified")
else:
    print(f"AUDIT PASSED — {result['anchors_verified']} anchors verified")

---

The verification above ran entirely against the Ethereum blockchain.
No AnchorRegistry servers involved.

**One ownership token proves the entire tree is yours.**

```
treeId          = keccak256(K || rootArId)    → tree ownership
tokenCommitment = keccak256(K || childArId)   → per-anchor initiation
```

Same ownership token `K`. Same construction. One key proves everything.

To dispute a tokenCommitment: produce the ownership token `K` such that `keccak256(K || arId) == tokenCommitment`. Producing this token proves you initiated the registration. The proof is on Ethereum — permanent and independent of AnchorRegistry infrastructure.